# FPGA DRAM bit-flip analysis

Loads every `experiment_*.csv` in `fpga_data/` and compares runs:

* **same pattern, different hold time** (`hold_s`, the time between refreshes)
* **same hold time, different pattern** (`00`, `FF`, `55`, `AA`, …)

Since the flips are thought to be mostly thermal, it also checks how flip counts depend on temperature,
and whether the run-to-run scatter looks like Poisson noise.

### Column meanings (best guess; check against the FPGA firmware)

| column | likely meaning |
|---|---|
| `hold_s` | time the DRAM is left without refresh before readout (s) |
| `refresh_rate` | refresh mode (`INTERNAL` = controller's own auto-refresh setting) |
| `pattern` | byte written to every address (hex) |
| `flip_count` | number of **words/addresses** that read back wrong |
| `bit_flip_count` | total number of **bits** that read back wrong (≥ `flip_count`) |
| `addr_count` | number of bad addresses actually logged (= `flip_count` when nothing was dropped) |
| `addr_complete` | `True` if every bad address fit in the log |
| `addr_overflow` | `True` if the address log overflowed (counts may then be truncated) |
| `temp_c` | temperature at readout (°C; probably the FPGA die sensor) |

So `bit_flip_count − flip_count` is the number of *extra* bits in words that had more than one bit flip.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_DIR   = Path("fpga_data")
DROP_FIRST = 0      # drop the first N iterations of each run (e.g. while the board warms up)

# One fixed colour per pattern. A colour belongs to the pattern, so it never changes between plots.
PATTERN_COLORS = {"00": "#2a78d6", "FF": "#eb6834", "55": "#1baf7a", "AA": "#eda100"}
EXTRA_COLORS   = ["#e87ba4", "#008300", "#6250d6", "#e34948"]   # for any new patterns
MARKERS        = ["o", "s", "^", "D", "v", "P", "X"]
LINESTYLES     = ["-", "--", ":", "-."]

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "axes.titleweight": "bold", "lines.linewidth": 1.6,
    "legend.frameon": False,
})

def pattern_color(p):
    if p not in PATTERN_COLORS:
        PATTERN_COLORS[p] = EXTRA_COLORS[(len(PATTERN_COLORS) - 4) % len(EXTRA_COLORS)]
    return PATTERN_COLORS[p]

## Load all runs

In [ ]:
def load_run(path):
    # The '# key: value' lines at the top hold run metadata
    meta = {}
    with open(path) as f:
        for line in f:
            if not line.startswith("#"):
                break
            k, _, v = line[1:].partition(":")
            meta[k.strip()] = v.strip()
    # pattern MUST be read as text, otherwise "00" -> 0 and "55" -> 55 (an int)
    df = pd.read_csv(path, comment="#", dtype={"pattern": str, "refresh_rate": str})
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601")
    df["pattern"]   = df["pattern"].str.upper().str.zfill(2)
    df["run"]       = meta.get("experiment", path.stem)
    df["file"]      = path.name
    t0 = pd.to_datetime(meta["start_time"]) if "start_time" in meta else df["timestamp"].iloc[0]
    df["elapsed_s"]  = (df["timestamp"] - t0).dt.total_seconds()
    df["readout_dt"] = df["timestamp"].diff().dt.total_seconds()      # real time between readouts
    df["extra_bits"] = df["bit_flip_count"] - df["flip_count"]       # bits beyond the first in each bad word
    df["bits_per_bad_word"] = df["bit_flip_count"] / df["flip_count"].replace(0, np.nan)
    return df.iloc[DROP_FIRST:]

files = sorted(DATA_DIR.glob("experiment_*.csv"))
data  = pd.concat([load_run(f) for f in files], ignore_index=True).sort_values("timestamp", ignore_index=True)
data["condition"] = data["pattern"] + " / " + data["hold_s"].astype(str) + " s"

print(f"{len(files)} files, {len(data)} readouts")
if data["addr_overflow"].any():
    print("WARNING: address log overflowed in these runs:", data.loc[data.addr_overflow, "run"].unique())
data.head()

## Summary table, one row per run

In [ ]:
def summarize(g):
    fc = g["flip_count"]
    return pd.Series({
        "pattern":     g["pattern"].iat[0],
        "hold_s":      g["hold_s"].iat[0],
        "refresh":     g["refresh_rate"].iat[0],
        "start":       g["timestamp"].iat[0].strftime("%Y-%m-%d %H:%M"),
        "n":           len(g),
        "words_mean":  fc.mean(),
        "words_std":   fc.std(ddof=1),
        "bits_mean":   g["bit_flip_count"].mean(),
        "bits/word":   g["bit_flip_count"].sum() / max(fc.sum(), 1),
        "fano":        fc.var(ddof=1) / fc.mean() if fc.mean() > 0 else np.nan,  # 1 for Poisson
        "T_mean":      g["temp_c"].mean(),
        "T_range":     g["temp_c"].max() - g["temp_c"].min(),
        "readout_dt":  g["readout_dt"].median(),
    })

summary = (data.groupby("run").apply(summarize, include_groups=False)
               .sort_values(["pattern", "hold_s", "start"]))
summary.style.format({"words_mean": "{:.1f}", "words_std": "{:.2f}", "bits_mean": "{:.1f}",
                      "bits/word": "{:.3f}", "fano": "{:.2f}", "T_mean": "{:.1f}",
                      "T_range": "{:.1f}", "readout_dt": "{:.2f}"})

**Reading the Fano factor:** if each readout were an independent Poisson count (random, unrelated events), variance ≈ mean, so Fano ≈ 1.
Fano well **below 1** means the flips come from a fixed set of weak cells that fail almost every time (binomial, not Poisson).
That is what retention-time (thermal) failures look like. Fano **above 1** means something is drifting during the run (usually temperature).

## 1. Time series for every run, one panel per pattern

In [ ]:
patterns = sorted(data["pattern"].unique())
fig, axes = plt.subplots(len(patterns), 2, figsize=(12, 2.8 * len(patterns)), sharex=True, squeeze=False)

for i, p in enumerate(patterns):
    sub = data[data.pattern == p]
    ax_f, ax_t = axes[i]
    for j, (run, g) in enumerate(sub.groupby("run", sort=False)):
        kw = dict(color=pattern_color(p), ls=LINESTYLES[j % 4], marker=MARKERS[j % 7], ms=3,
                  label=f"{run} (hold {g.hold_s.iat[0]} s)")
        ax_f.plot(g["elapsed_s"] / 60, g["flip_count"], **kw)
        ax_t.plot(g["elapsed_s"] / 60, g["temp_c"], **kw)
    ax_f.set_ylabel("bad words")
    ax_t.set_ylabel("temp (°C)")
    ax_f.set_title(f"pattern 0x{p}: flip count", loc="left")
    ax_t.set_title(f"pattern 0x{p}: temperature", loc="left")
    ax_f.legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("time since run start (min)")
fig.tight_layout()

plt.savefig('flips_vs_temp.png')

## 2. Same hold time, different patterns

In [ ]:
for hold, sub in data.groupby("hold_s"):
    pats = sorted(sub["pattern"].unique())
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # (a) distribution of bad words per readout
    ax = axes[0]
    for k, p in enumerate(pats):
        y = sub.loc[sub.pattern == p, "flip_count"]
        x = k + np.random.default_rng(0).uniform(-0.18, 0.18, len(y))
        ax.scatter(x, y, s=10, color=pattern_color(p), alpha=0.6, lw=0)
        ax.errorbar(k + 0.32, y.mean(), yerr=y.std(ddof=1), fmt="o", color="#333", ms=5, capsize=3)
    ax.set_xticks(range(len(pats)), [f"0x{p}" for p in pats])
    ax.set_ylabel("bad words per readout")
    ax.set_title(f"(a) flip count, hold = {hold} s", loc="left")

    # (b) multi-bit flips: bits per bad word
    ax = axes[1]
    for k, p in enumerate(pats):
        g = sub[sub.pattern == p]
        r = g["bit_flip_count"].sum() / g["flip_count"].sum()
        ax.bar(k, r - 1, bottom=1, color=pattern_color(p), width=0.6)
        ax.text(k, r, f"{r:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(len(pats)), [f"0x{p}" for p in pats])
    ax.set_ylim(1, None)
    ax.set_ylabel("flipped bits / bad word")
    ax.set_title("(b) multi-bit words (1 = all single-bit)", loc="left")

    # (c) flip count vs temperature: is the pattern difference just temperature?
    ax = axes[2]
    for p in pats:
        g = sub[sub.pattern == p]
        ax.scatter(g["temp_c"], g["flip_count"], s=12, color=pattern_color(p), label=f"0x{p}", lw=0)
    ax.set_yscale("log")
    ax.set_xlabel("temp (°C)")
    ax.set_ylabel("bad words (log)")
    ax.set_title("(c) vs temperature", loc="left")
    ax.legend()
    fig.tight_layout()

plt.savefig('flip_by_words.png')

A single byte value holds the same data in every bit only for `00` and `FF`. For `55` (01010101) and `AA` (10101010), neighbouring bits
hold opposite values. If cells failed independently, `55`/`AA` should land roughly halfway between `00` and `FF`.
Landing well above both means the *data pattern itself* matters, for example through coupling between adjacent cells or bitlines.

## 3. Same pattern, different hold times (retention curve)

In [ ]:
by_cond = (data.groupby(["pattern", "hold_s"])
               .agg(words=("flip_count", "mean"), words_sd=("flip_count", "std"),
                    bits=("bit_flip_count", "mean"), n=("flip_count", "size"),
                    T=("temp_c", "mean"))
               .reset_index())
by_cond["words_sem"] = by_cond["words_sd"] / np.sqrt(by_cond["n"])
display(by_cond)

multi = by_cond.groupby("pattern")["hold_s"].nunique()
if (multi < 2).all():
    print(f"Every pattern has only one hold time so far (hold_s = {sorted(data.hold_s.unique())}).\n"
          "Take runs at other hold times (e.g. 1, 2, 5, 10, 20, 60 s) and this cell will draw the retention curves.")
else:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for p, g in by_cond.groupby("pattern"):
        if g["hold_s"].nunique() < 2:
            continue
        g = g.sort_values("hold_s")
        ax.errorbar(g["hold_s"], g["words"], yerr=g["words_sem"], color=pattern_color(p),
                    marker="o", ms=5, capsize=3, label=f"0x{p}")
        # power-law fit N = A * t^k. Flip count grows as more weak cells' retention time drops below t
        ok = g["words"] > 0
        if ok.sum() >= 2:
            k, lnA = np.polyfit(np.log(g.loc[ok, "hold_s"]), np.log(g.loc[ok, "words"]), 1)
            tt = np.geomspace(g["hold_s"].min(), g["hold_s"].max(), 50)
            ax.plot(tt, np.exp(lnA) * tt**k, color=pattern_color(p), lw=1, ls=":")
            ax.annotate(f"k = {k:.2f}", (tt[-1], np.exp(lnA) * tt[-1]**k), fontsize=8,
                        xytext=(4, 0), textcoords="offset points", color="#555")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("hold time (s)"); ax.set_ylabel("mean bad words per readout")
    ax.set_title("Retention: flips vs hold time", loc="left")
    ax.legend()
    fig.tight_layout()

## 4. Temperature dependence

In [ ]:
# Leakage, and so the failure rate, rises roughly exponentially with T: N ∝ exp(T / T0).
# Fit ln(N) vs T for each (pattern, hold) and report the temperature rise that doubles the count.
rows = []
fig, ax = plt.subplots(figsize=(7, 4.5))
for (p, hold), g in data.groupby(["pattern", "hold_s"]):
    g = g[g.flip_count > 0]
    ax.scatter(g["temp_c"], g["flip_count"], s=10, color=pattern_color(p), lw=0, alpha=0.7)
    if g["temp_c"].max() - g["temp_c"].min() < 1.0:
        continue          # too little spread in T for a meaningful fit
    res = stats.linregress(g["temp_c"], np.log(g["flip_count"]))
    tt = np.linspace(g.temp_c.min(), g.temp_c.max(), 20)
    ax.plot(tt, np.exp(res.intercept + res.slope * tt), color=pattern_color(p), lw=1.2,
            label=f"0x{p}, {hold} s")
    rows.append({"pattern": p, "hold_s": hold, "slope_per_C": res.slope, "slope_err": res.stderr,
                 "doubling_dT_C": np.log(2) / res.slope if res.slope > 0 else np.nan,
                 "T_span": g.temp_c.max() - g.temp_c.min(), "r": res.rvalue})
ax.set_yscale("log")
ax.set_xlabel("temp (°C)"); ax.set_ylabel("bad words (log)")
ax.set_title("Flip count vs temperature (all runs)", loc="left")
ax.legend(fontsize=8)
fig.tight_layout()
pd.DataFrame(rows)

plt.savefig('flip_vs_temp_all_runs.png')

The fit is only meaningful where a run covers a few °C, which in practice means the warm-up runs.
When runs are mostly at a stable temperature the fit is dominated by other scatter, so use it as a rough guide only.
For a clean number, warm or cool the board on purpose (fan off/on) at a fixed pattern and hold time.

## 5. Counting statistics: Poisson or a fixed set of weak cells?

In [ ]:
# Take only readouts where T is within ±0.3 °C of the run median, so drift doesn't inflate the scatter
T_TOL = 0.3
st = data[(data.temp_c - data.groupby("run")["temp_c"].transform("median")).abs() <= T_TOL]
runs = summary.index.tolist()
ncol = 4
nrow = int(np.ceil(len(runs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.6 * nrow), squeeze=False)
for ax, run in zip(axes.flat, runs):
    g = st[st.run == run]["flip_count"]
    p = summary.loc[run, "pattern"]
    w = 3 * np.sqrt(g.mean())                     # show +/-3 sigma of the Poisson expectation
    lo, hi = int(min(g.min(), g.mean() - w)) - 1, int(max(g.max(), g.mean() + w)) + 2
    bins = np.arange(lo, hi) - 0.5
    ax.hist(g, bins=bins, color=pattern_color(p), alpha=0.85)
    k = np.arange(lo, hi)
    ax.plot(k, len(g) * stats.poisson.pmf(k, g.mean()), "k-", lw=1, label="Poisson, same mean")
    fano = g.var(ddof=1) / g.mean()
    ax.set_title(f"{run} (0x{p})", fontsize=9, loc="left")
    ax.text(0.97, 0.95, f"Fano={fano:.2f}\nn={len(g)}", transform=ax.transAxes, ha="right", va="top", fontsize=8)
for ax in axes.flat[len(runs):]:
    ax.set_visible(False)
axes.flat[0].legend(fontsize=7, loc="upper left")
fig.suptitle("Bad words per readout (steady-temperature readouts only)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()

## 6. Run-to-run reproducibility

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = summary.sort_values(["pattern", "start"])
x = np.arange(len(order))
for xi, (run, r) in zip(x, order.iterrows()):
    if xi%3==0:
        ax.errorbar(xi, r.words_mean, yerr=r.words_std, fmt="o", color=pattern_color(r.pattern), capsize=3, ms=6, label=r.pattern)
    else:
        ax.errorbar(xi, r.words_mean, yerr=r.words_std, fmt="o", color=pattern_color(r.pattern), capsize=3, ms=6)

ax.set_xticks(x, [f"{run.replace('experiment_', '')}\n{r.T_mean:.1f}°C" for run, r in order.iterrows()], fontsize=8)
ax.set_xlabel("run (mean temperature)")
ax.set_yscale("log")
ax.set_ylabel("mean bad words ± 1σ (log)")
ax.set_title("Mean flip count per run, grouped by pattern", loc="left")
fig.tight_layout()

fig.legend()

plt.savefig('mean_flip_count_by_word.png')


## Next steps toward a cosmic-ray signal

Thermal (retention) flips come back on the **same addresses** readout after readout. A cosmic-ray or radiation hit
should show up as an address that is *not* in the usual weak-cell set, often as a small cluster of neighbouring cells.
These CSVs only hold counts. To separate the two you need the bad-address list for every readout. Then you can:

1. Build a "weak-cell map" from a long baseline run (addresses that fail in, say, >5 % of readouts).
2. Subtract it from every readout. What's left are the candidate events.
3. Compare the candidate rate with the source in and out, or shielded and unshielded (or with a scintillator coincidence).

A longer hold time or a higher temperature brings in more weak cells and raises the background. Keeping the board cool and the hold time short
keeps the thermal background low. Whether radiation-induced upsets can be seen at all depends on the DRAM technology.